# Integrate screening signals

Combines concentration, recurrence, single-bidder, value, and network-position signals into a transparent review-prioritization workflow.

Run this notebook from the `notebooks/` directory after completing the preceding numbered stage. Generated files are written to the documented project directories.


# Integrate signals and characterize patterns


In [ ]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


In [ ]:
# Parametros

MIN_AWARD_AMOUNT_CONCENTRATION = 5_000
MIN_BUYER_AWARDS_CONCENTRATION = 5

RECURRENCE_WINDOW_DAYS = 90
RECURRENCE_MIN_PROCEDURES = 4  # equivalente a >3

SCREENING_PERCENTILE = 0.90

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [ ]:
# Load the processed datasets

DATA_PROCESSED = Path("../data/processed")
NEO4J_DIR = Path("../data/neo4j")
ANALYTICS_DIR = Path("../outputs/tables")
ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)

procedimientos_path = DATA_PROCESSED / "procedures_2025.csv"
participacion_path = DATA_PROCESSED / "tender_participation_2025.csv"
adjudicaciones_path = DATA_PROCESSED / "awards_2025.csv"
network_metrics_path = NEO4J_DIR / "actor_network_metrics_2025.csv"

for p in [
    procedimientos_path,
    participacion_path,
    adjudicaciones_path,
    network_metrics_path,
]:
    if not p.exists():
        raise FileNotFoundError(f"No se encontró el archivo requerido: {p.resolve()}")

dtype_common = {
    "ocid": "string",
    "buyer_id": "string",
    "supplier_id": "string",
    "tenderer_id": "string",
    "cpc_5": "string",
}

procedimientos_df = pd.read_csv(
    procedimientos_path,
    dtype={k: v for k, v in dtype_common.items() if k in ["ocid", "buyer_id"]},
)

participacion_df = pd.read_csv(
    participacion_path,
    dtype={k: v for k, v in dtype_common.items() if k in ["ocid", "tenderer_id"]},
)

adjudicaciones_df = pd.read_csv(
    adjudicaciones_path,
    dtype={
        "ocid": "string",
        "buyer_id": "string",
        "supplier_id": "string",
        "cpc_5": "string",
    },
)

network_metrics_df = pd.read_csv(
    network_metrics_path,
    dtype={"actor_id": "string"},
)

print("Procedimientos:", len(procedimientos_df))
print("Participaciones de oferentes:", len(participacion_df))
print("Adjudicaciones:", len(adjudicaciones_df))
print("Actores con métricas de red:", len(network_metrics_df))


In [ ]:
required_proc = {"ocid", "buyer_id", "number_of_tenderers"}
required_award = {
    "ocid",
    "buyer_id",
    "supplier_id",
    "award_amount",
    "award_date_local",
    "cpc_5",
}
required_network = {
    "actor_type",
    "actor_id",
    "degree",
    "strength_frequency",
    "strength_amount",
}

def assert_columns(df, required, df_name):
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"{df_name} no contiene las columnas requeridas: {sorted(missing)}"
        )

assert_columns(procedimientos_df, required_proc, "procedimientos_df")
assert_columns(adjudicaciones_df, required_award, "adjudicaciones_df")
assert_columns(network_metrics_df, required_network, "network_metrics_df")

if procedimientos_df["ocid"].duplicated().any():
    raise ValueError("La base de procedimientos contiene OCID duplicados.")

if adjudicaciones_df[["ocid", "supplier_id"]].isna().any().any():
    raise ValueError("Existen adjudicaciones sin OCID o supplier_id.")

print("Validaciones mínimas superadas.")


In [ ]:
# Prepare base variables

adjudicaciones_df["award_amount"] = pd.to_numeric(
    adjudicaciones_df["award_amount"],
    errors="coerce",
)

adjudicaciones_df["award_date_dt"] = pd.to_datetime(
    adjudicaciones_df["award_date_local"],
    errors="coerce",
)

procedimientos_df["number_of_tenderers"] = pd.to_numeric(
    procedimientos_df["number_of_tenderers"],
    errors="coerce",
)

# Columnas descriptivas preferidas.
buyer_name_col = (
    "buyer_name_normalized"
    if "buyer_name_normalized" in procedimientos_df.columns
    else "buyer_name"
)

supplier_name_col = (
    "nombre_proveedor_limpio"
    if "nombre_proveedor_limpio" in adjudicaciones_df.columns
    else "supplier_name"
)

print("Buyer name:", buyer_name_col)
print("Supplier name:", supplier_name_col)


In [ ]:
# Calculate concentration by contracting entity
# - Concentracion monetaria

def normalized_hhi(hhi, n):
    if pd.isna(hhi) or pd.isna(n) or n <= 1:
        return np.nan
    return (hhi - (1 / n)) / (1 - (1 / n))


def normalized_entropy(shares):
    shares = np.asarray(shares, dtype=float)
    shares = shares[shares > 0]

    n = len(shares)
    if n <= 1:
        return np.nan

    entropy = -(shares * np.log(shares)).sum()
    return entropy / np.log(n)


def concentration_by_buyer(
    awards,
    min_amount=5_000,
    min_buyer_awards=5,
):
    df = awards.loc[
        awards["award_amount"].notna()
        & (awards["award_amount"] >= min_amount)
    ].copy()

    buyer_counts = (
        df.groupby("buyer_id")["ocid"]
        .nunique()
        .rename("numero_adjudicaciones")
    )

    eligible_buyers = buyer_counts[
        buyer_counts >= min_buyer_awards
    ].index

    df = df[df["buyer_id"].isin(eligible_buyers)].copy()

    pair = (
        df.groupby(["buyer_id", "supplier_id"], as_index=False)
        .agg(
            frequency=("ocid", "nunique"),
            amount=("award_amount", "sum"),
        )
    )

    rows = []

    for buyer_id, g in pair.groupby("buyer_id"):
        n_suppliers = g["supplier_id"].nunique()
        total_freq = g["frequency"].sum()
        total_amount = g["amount"].sum()

        freq_shares = g["frequency"] / total_freq
        amount_shares = g["amount"] / total_amount

        hhi_freq = float((freq_shares ** 2).sum())
        hhi_amount = float((amount_shares ** 2).sum())

        cr4_amount = float(
            g.nlargest(4, "amount")["amount"].sum() / total_amount
        )

        rows.append({
            "buyer_id": buyer_id,
            "numero_adjudicaciones": int(total_freq),
            "numero_proveedores": int(n_suppliers),
            "monto_total": float(total_amount),
            "hhi_frecuencia": hhi_freq,
            "hhi_monetario": hhi_amount,
            "hhi_frecuencia_normalizado": normalized_hhi(
                hhi_freq, n_suppliers
            ),
            "hhi_monetario_normalizado": normalized_hhi(
                hhi_amount, n_suppliers
            ),
            "cr4_monetario": cr4_amount,
            "entropia_frecuencia_normalizada": normalized_entropy(
                freq_shares
            ),
            "entropia_monetaria_normalizada": normalized_entropy(
                amount_shares
            ),
        })

    return pd.DataFrame(rows), pair


concentration_df, buyer_supplier_concentration_df = concentration_by_buyer(
    adjudicaciones_df,
    min_amount=MIN_AWARD_AMOUNT_CONCENTRATION,
    min_buyer_awards=MIN_BUYER_AWARDS_CONCENTRATION,
)

print("Entidades en análisis de concentración:", len(concentration_df))
display(concentration_df.head())


In [ ]:
concentration_summary = concentration_df[
    [
        "hhi_frecuencia_normalizado",
        "hhi_monetario_normalizado",
        "cr4_monetario",
        "entropia_monetaria_normalizada",
    ]
].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
).T

display(concentration_summary)


In [ ]:
def recurrence_windows(
    awards,
    window_days=90,
    min_procedures=4,
):
    required = [
        "buyer_id",
        "supplier_id",
        "cpc_5",
        "ocid",
        "award_date_dt",
    ]

    df = awards[required].dropna().copy()

    df = (
        df.sort_values("award_date_dt")
        .drop_duplicates(
            ["buyer_id", "supplier_id", "cpc_5", "ocid"],
            keep="first",
        )
    )

    rows = []

    for (buyer_id, supplier_id, cpc_5), g in df.groupby(
        ["buyer_id", "supplier_id", "cpc_5"],
        sort=False,
    ):
        g = g.sort_values("award_date_dt").reset_index(drop=True)

        dates = g["award_date_dt"]
        ocids = g["ocid"]

        max_count = 0
        best_start = None
        best_end = None
        best_ocids = None

        left = 0

        for right in range(len(g)):
            while (
                dates.iloc[right] - dates.iloc[left]
            ).days > window_days:
                left += 1

            window_count = right - left + 1

            if window_count > max_count:
                max_count = window_count
                best_start = dates.iloc[left]
                best_end = dates.iloc[right]
                best_ocids = list(ocids.iloc[left:right + 1])

        rows.append({
            "buyer_id": buyer_id,
            "supplier_id": supplier_id,
            "cpc_5": cpc_5,
            "max_procesos_ventana": int(max_count),
            "recurrencia": int(max_count >= min_procedures),
            "ventana_inicio": best_start,
            "ventana_fin": best_end,
            "ocids_ventana": "|".join(best_ocids or []),
        })

    return pd.DataFrame(rows)


recurrence_df = recurrence_windows(
    adjudicaciones_df,
    window_days=RECURRENCE_WINDOW_DAYS,
    min_procedures=RECURRENCE_MIN_PROCEDURES,
)

print("Combinaciones Buyer-Supplier-CPC5:", len(recurrence_df))
print("Combinaciones recurrentes:", int(recurrence_df["recurrencia"].sum()))

display(
    recurrence_df[
        recurrence_df["recurrencia"] == 1
    ].sort_values(
        "max_procesos_ventana",
        ascending=False
    ).head(20)
)


In [ ]:
# Analyze bidder participation

procedure_participation_df = procedimientos_df[
    ["ocid", "buyer_id", "number_of_tenderers"]
].copy()

observed_participation = (
    procedure_participation_df["number_of_tenderers"].notna()
)

procedure_participation_df["low_participation_1"] = pd.Series(
    pd.NA,
    index=procedure_participation_df.index,
    dtype="Float64",
)

procedure_participation_df.loc[
    observed_participation,
    "low_participation_1",
] = (
    procedure_participation_df.loc[
        observed_participation,
        "number_of_tenderers",
    ].eq(1).astype(float)
)

procedure_participation_df[
    "low_participation_3_or_less"
] = pd.Series(
    pd.NA,
    index=procedure_participation_df.index,
    dtype="Float64",
)

procedure_participation_df.loc[
    observed_participation,
    "low_participation_3_or_less",
] = (
    procedure_participation_df.loc[
        observed_participation,
        "number_of_tenderers",
    ].le(3).astype(float)
)

participation_summary = (
    procedure_participation_df["number_of_tenderers"]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

display(participation_summary.to_frame("number_of_tenderers"))

print(
    "Procedimientos con participación observada:",
    int(observed_participation.sum()),
    "/",
    len(procedure_participation_df),
)


In [ ]:
award_participation_df = adjudicaciones_df[
    [
        "ocid",
        "buyer_id",
        "supplier_id",
        "cpc_5",
        "award_amount",
        "award_date_dt",
    ]
].merge(
    procedure_participation_df,
    on=["ocid", "buyer_id"],
    how="left",
)

relation_participation_df = (
    award_participation_df
    .groupby(["buyer_id", "supplier_id"], as_index=False)
    .agg(
        relation_procedures=("ocid", "nunique"),
        relation_total_amount=("award_amount", "sum"),
        tenderers_mean=("number_of_tenderers", "mean"),
        tenderers_median=("number_of_tenderers", "median"),
        participation_observed_procedures=(
            "number_of_tenderers",
            "count",
        ),
        single_bidder_share=("low_participation_1", "mean"),
        low_participation_share=(
            "low_participation_3_or_less",
            "mean",
        ),
        missing_participation=(
            "number_of_tenderers",
            lambda s: int(s.isna().sum()),
        ),
    )
)

relation_participation_df[
    "participation_coverage"
] = (
    relation_participation_df[
        "participation_observed_procedures"
    ]
    / relation_participation_df["relation_procedures"]
)

display(relation_participation_df.head())

print(
    "Relaciones sin información de participación:",
    int(
        relation_participation_df[
            "participation_observed_procedures"
        ].eq(0).sum()
    ),
)


In [ ]:
# Prepare and validate network metrics

required_network_columns = {
    "actor_type",
    "actor_id",
    "degree",
    "strength_frequency",
    "strength_amount",
    "betweenness",
}

missing_network_columns = (
    required_network_columns - set(network_metrics_df.columns)
)

if missing_network_columns:
    raise ValueError(
        "actor_network_metrics_2025.csv no contiene las columnas "
        f"requeridas: {sorted(missing_network_columns)}. "
        "Regenera la exportación desde el Notebook 05."
    )

for col in [
    "degree",
    "strength_frequency",
    "strength_amount",
    "betweenness",
]:
    network_metrics_df[col] = pd.to_numeric(
        network_metrics_df[col],
        errors="coerce",
    )

betweenness_non_null = int(
    network_metrics_df["betweenness"].notna().sum()
)

print(
    "Actores con betweenness disponible:",
    betweenness_non_null,
    "/",
    len(network_metrics_df),
)

display(
    network_metrics_df[
        [
            "degree",
            "strength_frequency",
            "strength_amount",
            "betweenness",
        ]
    ].describe()
)

if betweenness_non_null == 0:
    raise ValueError(
        "La columna 'betweenness' existe, pero todos sus valores son nulos. "
        "El Notebook 06 no continuará silenciosamente sin esta métrica. "
        "Revisa el cálculo y la exportación desde el Notebook 05."
    )

buyer_network_df = (
    network_metrics_df[
        network_metrics_df["actor_type"].eq("Buyer")
    ]
    .rename(columns={
        "actor_id": "buyer_id",
        "actor_name": "buyer_name_network",
        "degree": "buyer_degree",
        "strength_frequency": "buyer_strength_frequency",
        "strength_amount": "buyer_strength_amount",
        "betweenness": "buyer_betweenness",
    })
)

supplier_network_df = (
    network_metrics_df[
        network_metrics_df["actor_type"].eq("Supplier")
    ]
    .rename(columns={
        "actor_id": "supplier_id",
        "actor_name": "supplier_name_network",
        "degree": "supplier_degree",
        "strength_frequency": "supplier_strength_frequency",
        "strength_amount": "supplier_strength_amount",
        "betweenness": "supplier_betweenness",
    })
)

buyer_network_keep = [
    c for c in [
        "buyer_id",
        "buyer_name_network",
        "buyer_degree",
        "buyer_strength_frequency",
        "buyer_strength_amount",
        "buyer_betweenness",
    ]
    if c in buyer_network_df.columns
]

supplier_network_keep = [
    c for c in [
        "supplier_id",
        "supplier_name_network",
        "supplier_degree",
        "supplier_strength_frequency",
        "supplier_strength_amount",
        "supplier_betweenness",
    ]
    if c in supplier_network_df.columns
]

buyer_network_df = buyer_network_df[
    buyer_network_keep
].copy()

supplier_network_df = supplier_network_df[
    supplier_network_keep
].copy()

print("Buyers con métricas:", len(buyer_network_df))
print("Suppliers con métricas:", len(supplier_network_df))


In [ ]:
# Base integrada Buyer-Supplier.

relation_base_df = (
    adjudicaciones_df
    .groupby(["buyer_id", "supplier_id"], as_index=False)
    .agg(
        relation_frequency=("ocid", "nunique"),
        relation_amount=("award_amount", "sum"),
        relation_first_award=("award_date_dt", "min"),
        relation_last_award=("award_date_dt", "max"),
        relation_cpc5_count=("cpc_5", "nunique"),
    )
)

relation_dated_procedures_df = (
    adjudicaciones_df[
        adjudicaciones_df["award_date_dt"].notna()
    ]
    .groupby(
        ["buyer_id", "supplier_id"],
        as_index=False,
    )
    .agg(
        relation_procedures_with_date=("ocid", "nunique")
    )
)

relation_base_df = relation_base_df.merge(
    relation_dated_procedures_df,
    on=["buyer_id", "supplier_id"],
    how="left",
)

relation_base_df[
    "relation_procedures_with_date"
] = (
    relation_base_df["relation_procedures_with_date"]
    .fillna(0)
    .astype(int)
)

relation_base_df[
    "award_date_coverage"
] = (
    relation_base_df["relation_procedures_with_date"]
    / relation_base_df["relation_frequency"]
)

relation_recurrence_df = (
    recurrence_df
    .groupby(["buyer_id", "supplier_id"], as_index=False)
    .agg(
        recurrent_cpc5_count=("recurrencia", "sum"),
        max_procesos_90d=("max_procesos_ventana", "max"),
        recurrence_any=("recurrencia", "max"),
    )
)

relation_integrated_df = (
    relation_base_df
    .merge(
        relation_participation_df,
        on=["buyer_id", "supplier_id"],
        how="left",
    )
    .merge(
        relation_recurrence_df,
        on=["buyer_id", "supplier_id"],
        how="left",
    )
    .merge(
        concentration_df,
        on="buyer_id",
        how="left",
    )
    .merge(
        buyer_network_df,
        on="buyer_id",
        how="left",
    )
    .merge(
        supplier_network_df,
        on="supplier_id",
        how="left",
    )
)

relation_integrated_df["recurrence_any"] = (
    relation_integrated_df["recurrence_any"]
    .fillna(0)
    .astype(int)
)

relation_integrated_df["recurrent_cpc5_count"] = (
    relation_integrated_df["recurrent_cpc5_count"]
    .fillna(0)
    .astype(int)
)

relation_integrated_df[
    "recurrence_available"
] = (
    relation_integrated_df["relation_frequency"].lt(
        RECURRENCE_MIN_PROCEDURES
    )
    | relation_integrated_df["award_date_coverage"].eq(1.0)
)

print("Relaciones integradas:", len(relation_integrated_df))
print(
    "Relaciones con recurrencia evaluable:",
    int(relation_integrated_df["recurrence_available"].sum()),
    "/",
    len(relation_integrated_df),
)
display(relation_integrated_df.head())


In [ ]:
expected_edges = (
    adjudicaciones_df[["buyer_id", "supplier_id"]]
    .drop_duplicates()
    .shape[0]
)

if len(relation_integrated_df) != expected_edges:
    raise ValueError(
        "La base integrada no conserva una fila por relación Buyer-Supplier."
    )

expected_frequency = adjudicaciones_df["ocid"].nunique()

relation_frequency_sum = relation_integrated_df[
    "relation_frequency"
].sum()

print("Relaciones Buyer-Supplier esperadas:", expected_edges)
print("Suma de frecuencias por relación:", relation_frequency_sum)
print("Adjudicaciones registradas:", len(adjudicaciones_df))

print(
    "Monto integrado:",
    relation_integrated_df["relation_amount"].sum()
)
print(
    "Monto adjudicaciones:",
    adjudicaciones_df["award_amount"].sum()
)


In [ ]:
def distribution_row(df, col, unit):
    if col not in df.columns:
        return None

    s = pd.to_numeric(
        df[col],
        errors="coerce",
    ).dropna()

    if s.empty:
        return None

    return {
        "variable": col,
        "unidad_analisis": unit,
        "n_observado": int(s.shape[0]),
        "p50": s.quantile(0.50),
        "p75": s.quantile(0.75),
        "p90": s.quantile(0.90),
        "p95": s.quantile(0.95),
        "p99": s.quantile(0.99),
    }


supplier_unique_df = (
    supplier_network_df
    .drop_duplicates("supplier_id")
    .copy()
)

buyer_unique_df = (
    buyer_network_df
    .drop_duplicates("buyer_id")
    .copy()
)

screening_specs = [
    (
        concentration_df,
        "hhi_monetario_normalizado",
        "Buyer elegible para concentración",
    ),
    (
        concentration_df,
        "hhi_frecuencia_normalizado",
        "Buyer elegible para concentración",
    ),
    (
        supplier_unique_df,
        "supplier_degree",
        "Supplier único",
    ),
    (
        supplier_unique_df,
        "supplier_strength_frequency",
        "Supplier único",
    ),
    (
        supplier_unique_df,
        "supplier_strength_amount",
        "Supplier único",
    ),
    (
        supplier_unique_df,
        "supplier_betweenness",
        "Supplier único",
    ),
    (
        buyer_unique_df,
        "buyer_degree",
        "Buyer único",
    ),
    (
        buyer_unique_df,
        "buyer_betweenness",
        "Buyer único",
    ),
    (
        relation_integrated_df,
        "relation_amount",
        "Relación Buyer-Supplier",
    ),
    (
        relation_integrated_df,
        "relation_frequency",
        "Relación Buyer-Supplier",
    ),
    (
        relation_integrated_df,
        "single_bidder_share",
        "Relación Buyer-Supplier con participación observada",
    ),
    (
        relation_integrated_df,
        "low_participation_share",
        "Relación Buyer-Supplier con participación observada",
    ),
]

threshold_rows = []

for df, col, unit in screening_specs:
    row = distribution_row(df, col, unit)
    if row is not None:
        threshold_rows.append(row)

screening_thresholds_df = pd.DataFrame(
    threshold_rows
)

display(screening_thresholds_df)


In [ ]:
def percentile_threshold(df, col, q=0.90):
    s = pd.to_numeric(
        df[col],
        errors="coerce",
    ).dropna()

    return (
        s.quantile(q)
        if not s.empty
        else np.nan
    )


hhi_amount_thr = percentile_threshold(
    concentration_df,
    "hhi_monetario_normalizado",
    SCREENING_PERCENTILE,
)

supplier_degree_thr = percentile_threshold(
    supplier_unique_df,
    "supplier_degree",
    SCREENING_PERCENTILE,
)

supplier_betweenness_thr = percentile_threshold(
    supplier_unique_df,
    "supplier_betweenness",
    SCREENING_PERCENTILE,
)

relation_amount_thr = percentile_threshold(
    relation_integrated_df,
    "relation_amount",
    SCREENING_PERCENTILE,
)

single_bidder_p90 = percentile_threshold(
    relation_integrated_df,
    "single_bidder_share",
    SCREENING_PERCENTILE,
)

SINGLE_BIDDER_SHARE_THRESHOLD = 0.50

# <=3 oferentes continúa como dimensión contextual.
limited_participation_p90 = percentile_threshold(
    relation_integrated_df,
    "low_participation_share",
    SCREENING_PERCENTILE,
)

thresholds_used_df = pd.DataFrame([
    {
        "variable_o_regla": "hhi_monetario_normalizado_p90",
        "unidad_analisis": "Buyer elegible para concentración",
        "n_observado": int(
            concentration_df[
                "hhi_monetario_normalizado"
            ].notna().sum()
        ),
        "umbral_exploratorio": hhi_amount_thr,
    },
    {
        "variable_o_regla": "supplier_degree_p90",
        "unidad_analisis": "Supplier único",
        "n_observado": int(
            supplier_unique_df[
                "supplier_degree"
            ].notna().sum()
        ),
        "umbral_exploratorio": supplier_degree_thr,
    },
    {
        "variable_o_regla": "supplier_betweenness_p90",
        "unidad_analisis": "Supplier único",
        "n_observado": int(
            supplier_unique_df[
                "supplier_betweenness"
            ].notna().sum()
        ),
        "umbral_exploratorio": supplier_betweenness_thr,
    },
    {
        "variable_o_regla": "relation_amount_p90",
        "unidad_analisis": "Relación Buyer-Supplier",
        "n_observado": int(
            relation_integrated_df[
                "relation_amount"
            ].notna().sum()
        ),
        "umbral_exploratorio": relation_amount_thr,
    },
    {
        "variable_o_regla": "single_bidder_share_p90_referencia",
        "unidad_analisis": (
            "Relación Buyer-Supplier con participación observada"
        ),
        "n_observado": int(
            relation_integrated_df[
                "single_bidder_share"
            ].notna().sum()
        ),
        "umbral_exploratorio": single_bidder_p90,
    },
    {
        "variable_o_regla": "single_bidder_share_umbral_principal",
        "unidad_analisis": (
            "Relación Buyer-Supplier con participación observada"
        ),
        "n_observado": int(
            relation_integrated_df[
                "single_bidder_share"
            ].notna().sum()
        ),
        "umbral_exploratorio": SINGLE_BIDDER_SHARE_THRESHOLD,
    },
    {
        "variable_o_regla": "low_participation_share_p90_contexto",
        "unidad_analisis": (
            "Relación Buyer-Supplier con participación observada"
        ),
        "n_observado": int(
            relation_integrated_df[
                "low_participation_share"
            ].notna().sum()
        ),
        "umbral_exploratorio": limited_participation_p90,
    },
])

display(thresholds_used_df)

if pd.isna(supplier_betweenness_thr):
    raise ValueError(
        "No fue posible calcular un umbral para supplier_betweenness. "
        "Revisa actor_network_metrics_2025.csv antes de continuar."
    )

print(
    "P90 HHI monetario (Buyer-level):",
    hhi_amount_thr,
)
print(
    "P90 supplier degree (Supplier-level):",
    supplier_degree_thr,
)
print(
    "P90 supplier betweenness (Supplier-level):",
    supplier_betweenness_thr,
)


In [ ]:
ri = relation_integrated_df.copy()


ri["available_monetary_concentration"] = (
    ri["hhi_monetario_normalizado"].notna()
)

ri["available_recurrence"] = (
    ri["recurrence_available"].fillna(False)
)

ri["available_single_bidder"] = (
    ri["single_bidder_share"].notna()
)

ri["available_supplier_degree"] = (
    ri["supplier_degree"].notna()
)

ri["available_supplier_betweenness"] = (
    ri["supplier_betweenness"].notna()
)

ri["available_relation_amount"] = (
    ri["relation_amount"].notna()
)

# Señales

ri["signal_high_monetary_concentration"] = (
    ri["available_monetary_concentration"]
    & (
        ri["hhi_monetario_normalizado"]
        >= hhi_amount_thr
    )
)

ri["signal_recurrence"] = (
    ri["available_recurrence"]
    & (ri["recurrence_any"] == 1)
)

ri["signal_single_bidder"] = (
    ri["available_single_bidder"]
    & (
        ri["single_bidder_share"]
        >= SINGLE_BIDDER_SHARE_THRESHOLD
    )
)

# NO entra al conteo principal.
ri["signal_limited_participation_context"] = (
    ri["low_participation_share"].notna()
    & (
        ri["low_participation_share"]
        >= limited_participation_p90
    )
)

ri["signal_supplier_high_degree"] = (
    ri["available_supplier_degree"]
    & (
        ri["supplier_degree"]
        >= supplier_degree_thr
    )
)

ri["signal_supplier_high_betweenness"] = (
    ri["available_supplier_betweenness"]
    & (
        ri["supplier_betweenness"]
        >= supplier_betweenness_thr
    )
)

ri["signal_supplier_structural_relevance"] = (
    ri["signal_supplier_high_degree"]
    | ri["signal_supplier_high_betweenness"]
)

ri["signal_high_relation_amount"] = (
    ri["available_relation_amount"]
    & (
        ri["relation_amount"]
        >= relation_amount_thr
    )
)

# Cruces interpretables.
ri["cross_concentration_recurrence"] = (
    ri["signal_high_monetary_concentration"]
    & ri["signal_recurrence"]
)

ri["cross_concentration_single_bidder"] = (
    ri["signal_high_monetary_concentration"]
    & ri["signal_single_bidder"]
)

ri["cross_recurrence_single_bidder"] = (
    ri["signal_recurrence"]
    & ri["signal_single_bidder"]
)

ri["cross_recurrence_supplier_centrality"] = (
    ri["signal_recurrence"]
    & ri["signal_supplier_structural_relevance"]
)

ri["cross_concentration_supplier_centrality"] = (
    ri["signal_high_monetary_concentration"]
    & ri["signal_supplier_structural_relevance"]
)

ri["cross_amount_recurrence"] = (
    ri["signal_high_relation_amount"]
    & ri["signal_recurrence"]
)

cross_cols = [
    "cross_concentration_recurrence",
    "cross_concentration_single_bidder",
    "cross_recurrence_single_bidder",
    "cross_recurrence_supplier_centrality",
    "cross_concentration_supplier_centrality",
    "cross_amount_recurrence",
]

cross_summary_df = pd.DataFrame({
    "cruce": cross_cols,
    "relaciones": [
        int(ri[c].sum())
        for c in cross_cols
    ],
})

cross_summary_df["porcentaje_relaciones"] = (
    cross_summary_df["relaciones"]
    / len(ri)
    * 100
)

display(cross_summary_df)


In [ ]:
base_signal_cols = [
    "signal_high_monetary_concentration",
    "signal_recurrence",
    "signal_single_bidder",
    "signal_supplier_high_degree",
    "signal_supplier_high_betweenness",
    "signal_high_relation_amount",
]

availability_cols = [
    "available_monetary_concentration",
    "available_recurrence",
    "available_single_bidder",
    "available_supplier_degree",
    "available_supplier_betweenness",
    "available_relation_amount",
]

ri["n_signals_available"] = (
    ri[availability_cols]
    .fillna(False)
    .astype(int)
    .sum(axis=1)
)

ri["signal_count"] = (
    ri[base_signal_cols]
    .fillna(False)
    .astype(int)
    .sum(axis=1)
)

ri["complete_signal_coverage"] = (
    ri["n_signals_available"] == len(base_signal_cols)
)

signal_count_summary = (
    ri["signal_count"]
    .value_counts()
    .sort_index()
    .rename_axis("numero_senales")
    .reset_index(name="relaciones")
)

signal_count_summary["porcentaje"] = (
    signal_count_summary["relaciones"]
    / len(ri)
    * 100
)

coverage_summary_df = pd.DataFrame({
    "dimension": [
        "concentracion_monetaria",
        "recurrencia",
        "single_bidder",
        "supplier_degree",
        "supplier_betweenness",
        "monto_relacion",
        "cobertura_completa_6_dimensiones",
    ],
    "relaciones_disponibles": [
        int(ri["available_monetary_concentration"].sum()),
        int(ri["available_recurrence"].sum()),
        int(ri["available_single_bidder"].sum()),
        int(ri["available_supplier_degree"].sum()),
        int(ri["available_supplier_betweenness"].sum()),
        int(ri["available_relation_amount"].sum()),
        int(ri["complete_signal_coverage"].sum()),
    ],
})

coverage_summary_df["porcentaje_relaciones"] = (
    coverage_summary_df["relaciones_disponibles"]
    / len(ri)
    * 100
)

display(signal_count_summary)
display(coverage_summary_df)

print(
    "Relaciones con convergencia multidimensional (>=3 señales):",
    int((ri["signal_count"] >= 3).sum()),
)

print(
    "Relaciones con cobertura completa de las 6 dimensiones:",
    int(ri["complete_signal_coverage"].sum()),
    "/",
    len(ri),
)


In [ ]:
single_bidder_sensitivity_rules = [
    {
        "escenario": "share >= 0.50",
        "threshold": 0.50,
        "operator": ">=",
    },
    {
        "escenario": "share >= 0.75",
        "threshold": 0.75,
        "operator": ">=",
    },
    {
        "escenario": "share = 1.00",
        "threshold": 1.00,
        "operator": "==",
    },
]

single_bidder_sensitivity_rows = []

for rule in single_bidder_sensitivity_rules:
    if rule["operator"] == ">=":
        mask = (
            ri["single_bidder_share"].notna()
            & (
                ri["single_bidder_share"]
                >= rule["threshold"]
            )
        )
    else:
        mask = (
            ri["single_bidder_share"].notna()
            & np.isclose(
                ri["single_bidder_share"],
                rule["threshold"],
            )
        )

    scenario_df = ri.loc[mask].copy()

    scenario_signal_count = (
        ri[
            [
                "signal_high_monetary_concentration",
                "signal_recurrence",
                "signal_supplier_high_degree",
                "signal_supplier_high_betweenness",
                "signal_high_relation_amount",
            ]
        ]
        .fillna(False)
        .astype(int)
        .sum(axis=1)
        + mask.astype(int)
    )

    single_bidder_sensitivity_rows.append(
        {
            "escenario": rule["escenario"],
            "umbral": rule["threshold"],
            "relaciones_single_bidder": int(mask.sum()),
            "pct_relaciones_single_bidder": (
                mask.mean() * 100
            ),
            "relaciones_convergencia_ge3": int(
                (scenario_signal_count >= 3).sum()
            ),
            "pct_convergencia_ge3": (
                (scenario_signal_count >= 3).mean()
                * 100
            ),
            "relaciones_frecuencia_1": int(
                scenario_df["relation_frequency"]
                .eq(1)
                .sum()
            ),
            "relaciones_frecuencia_2": int(
                scenario_df["relation_frequency"]
                .eq(2)
                .sum()
            ),
            "relaciones_frecuencia_3_mas": int(
                scenario_df["relation_frequency"]
                .ge(3)
                .sum()
            ),
        }
    )

single_bidder_sensitivity_df = pd.DataFrame(
    single_bidder_sensitivity_rows
)

display(single_bidder_sensitivity_df)

frequency_bands = pd.cut(
    ri["relation_frequency"],
    bins=[0, 1, 2, np.inf],
    labels=[
        "1 procedimiento",
        "2 procedimientos",
        "3 o más procedimientos",
    ],
)

single_bidder_frequency_df = (
    ri.assign(
        frequency_band=frequency_bands,
        single_bidder_50=(
            ri["single_bidder_share"].notna()
            & (ri["single_bidder_share"] >= 0.50)
        ),
        single_bidder_75=(
            ri["single_bidder_share"].notna()
            & (ri["single_bidder_share"] >= 0.75)
        ),
        single_bidder_100=(
            ri["single_bidder_share"].notna()
            & np.isclose(
                ri["single_bidder_share"],
                1.0,
            )
        ),
    )
    .groupby(
        "frequency_band",
        observed=False,
    )
    .agg(
        relaciones=("buyer_id", "size"),
        single_bidder_50=("single_bidder_50", "sum"),
        single_bidder_75=("single_bidder_75", "sum"),
        single_bidder_100=("single_bidder_100", "sum"),
    )
    .reset_index()
)

for col in [
    "single_bidder_50",
    "single_bidder_75",
    "single_bidder_100",
]:
    single_bidder_frequency_df[
        f"{col}_pct"
    ] = (
        single_bidder_frequency_df[col]
        / single_bidder_frequency_df["relaciones"]
        * 100
    )

display(single_bidder_frequency_df)

print(
    "La señal principal se mantiene en single_bidder_share >= 0.50 "
    "para conservar comparabilidad con el análisis principal."
)


In [ ]:
total_procedures = int(
    procedimientos_df["ocid"].nunique()
)

awarded_procedures = int(
    adjudicaciones_df["ocid"].nunique()
)

# Relaciones Buyer-Supplier únicas.
total_relations = int(
    len(ri)
)

complete_coverage_relations = int(
    ri["complete_signal_coverage"].sum()
)

convergent_relations = int(
    (ri["signal_count"] >= 3).sum()
)

high_convergence_relations = int(
    (ri["signal_count"] >= 4).sum()
)

# Casos exploratorios finales seleccionados.
prioritized_cases = int(
    len(case_candidates_df)
    if "case_candidates_df" in globals()
    else 5
)

prioritization_funnel_df = pd.DataFrame([
    {
        "etapa": "Procedimientos SIE procesados",
        "unidad": "Procedimiento",
        "cantidad": total_procedures,
    },
    {
        "etapa": "Procedimientos con adjudicación",
        "unidad": "Procedimiento",
        "cantidad": awarded_procedures,
    },
    {
        "etapa": "Relaciones Buyer-Supplier",
        "unidad": "Relación",
        "cantidad": total_relations,
    },
    {
        "etapa": "Relaciones con cobertura completa",
        "unidad": "Relación",
        "cantidad": complete_coverage_relations,
    },
    {
        "etapa": "Relaciones con >=3 señales",
        "unidad": "Relación",
        "cantidad": convergent_relations,
    },
    {
        "etapa": "Relaciones con >=4 señales",
        "unidad": "Relación",
        "cantidad": high_convergence_relations,
    },
    {
        "etapa": "Casos exploratorios priorizados",
        "unidad": "Relación",
        "cantidad": prioritized_cases,
    },
])

prioritization_funnel_df[
    "pct_sobre_relaciones"
] = np.where(
    prioritization_funnel_df["unidad"].eq("Relación"),
    (
        prioritization_funnel_df["cantidad"]
        / total_relations
        * 100
    ),
    np.nan,
)

prioritization_funnel_df[
    "pct_sobre_procedimientos"
] = np.where(
    prioritization_funnel_df["unidad"].eq("Procedimiento"),
    (
        prioritization_funnel_df["cantidad"]
        / total_procedures
        * 100
    ),
    np.nan,
)

display(prioritization_funnel_df)

print(
    f"De {total_relations:,} relaciones Buyer-Supplier, "
    f"{convergent_relations:,} presentan >=3 señales "
    f"({convergent_relations / total_relations * 100:.2f}%)."
)

print(
    f"{complete_coverage_relations:,} relaciones "
    f"({complete_coverage_relations / total_relations * 100:.2f}%) "
    "cuentan con cobertura completa de las seis dimensiones."
)


In [ ]:
top_convergence_df = (
    ri.sort_values(
        [
            "signal_count",
            "n_signals_available",
            "recurrence_any",
            "relation_amount",
            "supplier_degree",
            "supplier_betweenness",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            False,
            False,
        ],
    )
    .head(30)
)

display_cols = [
    "buyer_id",
    "buyer_name_network",
    "supplier_id",
    "supplier_name_network",
    "relation_frequency",
    "relation_amount",
    "max_procesos_90d",
    "recurrence_any",
    "single_bidder_share",
    "participation_coverage",
    "hhi_monetario_normalizado",
    "hhi_frecuencia_normalizado",
    "supplier_degree",
    "supplier_betweenness",
    "n_signals_available",
    "signal_count",
    "complete_signal_coverage",
]

display(
    top_convergence_df[
        [
            c
            for c in display_cols
            if c in top_convergence_df.columns
        ]
    ]
)


In [ ]:
concentration_sensitivity_rows = []

for min_amount in [0, 1_000, 5_000, 10_000]:
    cdf, _ = concentration_by_buyer(
        adjudicaciones_df,
        min_amount=min_amount,
        min_buyer_awards=MIN_BUYER_AWARDS_CONCENTRATION,
    )

    concentration_sensitivity_rows.append({
        "min_award_amount": min_amount,
        "buyers": len(cdf),
        "median_hhi_freq_norm": (
            cdf["hhi_frecuencia_normalizado"].median()
            if not cdf.empty else np.nan
        ),
        "median_hhi_amount_norm": (
            cdf["hhi_monetario_normalizado"].median()
            if not cdf.empty else np.nan
        ),
        "p90_hhi_amount_norm": (
            cdf["hhi_monetario_normalizado"].quantile(0.90)
            if not cdf.empty else np.nan
        ),
    })

concentration_sensitivity_df = pd.DataFrame(
    concentration_sensitivity_rows
)

display(concentration_sensitivity_df)


In [ ]:
recurrence_sensitivity_rows = []

for window in [30, 60, 90, 180]:
    for min_proc in [3, 4, 5]:
        rdf = recurrence_windows(
            adjudicaciones_df,
            window_days=window,
            min_procedures=min_proc,
        )

        recurrence_sensitivity_rows.append({
            "window_days": window,
            "min_procedures": min_proc,
            "recurrent_combinations": int(
                rdf["recurrencia"].sum()
            ),
            "total_combinations": len(rdf),
            "recurrent_share": (
                rdf["recurrencia"].mean()
                if len(rdf) else np.nan
            ),
        })

recurrence_sensitivity_df = pd.DataFrame(
    recurrence_sensitivity_rows
)

display(recurrence_sensitivity_df)


In [ ]:
ri["pattern_economic_concentration"] = (
    ri["signal_high_monetary_concentration"]
)

ri["pattern_temporal_recurrence"] = (
    ri["signal_recurrence"]
)

ri["pattern_single_bidder_relevance"] = (
    ri["signal_single_bidder"]
)

ri["pattern_limited_participation_context"] = (
    ri["signal_limited_participation_context"]
)

ri["pattern_structural_relevance"] = (
    ri["signal_supplier_structural_relevance"]
)

ri["pattern_multidimensional_convergence"] = (
    ri["signal_count"] >= 3
)

pattern_cols = [
    "pattern_economic_concentration",
    "pattern_temporal_recurrence",
    "pattern_single_bidder_relevance",
    "pattern_limited_participation_context",
    "pattern_structural_relevance",
    "pattern_multidimensional_convergence",
]

pattern_summary_df = pd.DataFrame({
    "patron": pattern_cols,
    "relaciones": [
        int(ri[c].sum())
        for c in pattern_cols
    ],
})

pattern_summary_df["porcentaje"] = (
    pattern_summary_df["relaciones"]
    / len(ri)
    * 100
)

display(pattern_summary_df)


In [ ]:
plot_df = signal_count_summary.copy()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    plot_df["numero_senales"].astype(str),
    plot_df["relaciones"],
)
ax.set_title(
    "Distribución de relaciones según número de señales analíticas"
)
ax.set_xlabel("Número de señales")
ax.set_ylabel("Relaciones Buyer-Supplier")
plt.tight_layout()
plt.show()


In [ ]:
# HHI monetario vs frecuencia, resaltando recurrencia.

plot_hhi = ri[
    [
        "hhi_frecuencia_normalizado",
        "hhi_monetario_normalizado",
        "recurrence_any",
    ]
].dropna()

fig, ax = plt.subplots(figsize=(8, 6))

for recurrence_value, group in plot_hhi.groupby("recurrence_any"):
    ax.scatter(
        group["hhi_frecuencia_normalizado"],
        group["hhi_monetario_normalizado"],
        alpha=0.55,
        label=(
            "Recurrente"
            if recurrence_value == 1
            else "No recurrente"
        ),
    )

ax.set_title(
    "Concentración por frecuencia vs concentración monetaria"
)
ax.set_xlabel("HHI frecuencia normalizado")
ax.set_ylabel("HHI monetario normalizado")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
plot_structural = ri[
    [
        "relation_amount",
        "supplier_degree",
        "recurrence_any",
    ]
].dropna()

plot_structural = plot_structural[
    plot_structural["relation_amount"] > 0
].copy()

fig, ax = plt.subplots(figsize=(8, 6))

for recurrence_value, group in plot_structural.groupby("recurrence_any"):
    ax.scatter(
        np.log10(group["relation_amount"]),
        group["supplier_degree"],
        alpha=0.5,
        label=(
            "Recurrente"
            if recurrence_value == 1
            else "No recurrente"
        ),
    )

ax.set_title(
    "Monto de la relación y diversidad de entidades del proveedor"
)
ax.set_xlabel("log10(monto acumulado de la relación)")
ax.set_ylabel("Degree del proveedor")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
#

case_candidates_df = (
    ri.sort_values(
        [
            "signal_count",
            "n_signals_available",
            "recurrence_any",
            "relation_amount",
            "supplier_degree",
            "supplier_betweenness",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            False,
            False,
        ],
    )
    .drop_duplicates(
        ["buyer_id", "supplier_id"]
    )
    .head(5)
    .copy()
)

case_cols = [
    "buyer_id",
    "buyer_name_network",
    "supplier_id",
    "supplier_name_network",
    "relation_frequency",
    "relation_amount",
    "max_procesos_90d",
    "recurrent_cpc5_count",
    "single_bidder_share",
    "participation_coverage",
    "hhi_monetario_normalizado",
    "hhi_frecuencia_normalizado",
    "supplier_degree",
    "supplier_strength_frequency",
    "supplier_strength_amount",
    "supplier_betweenness",
    "n_signals_available",
    "signal_count",
    "complete_signal_coverage",
]

display(
    case_candidates_df[
        [
            c
            for c in case_cols
            if c in case_candidates_df.columns
        ]
    ]
)


In [ ]:
case_keys = case_candidates_df[
    ["buyer_id", "supplier_id"]
].drop_duplicates()

case_procedures_df = (
    adjudicaciones_df
    .merge(
        case_keys,
        on=["buyer_id", "supplier_id"],
        how="inner",
    )
    .merge(
        procedure_participation_df[
            ["ocid", "number_of_tenderers"]
        ],
        on="ocid",
        how="left",
    )
    .sort_values(
        ["buyer_id", "supplier_id", "award_date_dt"]
    )
)

case_procedure_cols = [
    "buyer_id",
    "supplier_id",
    "ocid",
    "award_date_local",
    "award_amount",
    "cpc_5",
    "number_of_tenderers",
]

display(
    case_procedures_df[
        [c for c in case_procedure_cols if c in case_procedures_df.columns]
    ].head(100)
)


In [ ]:
# Export results.

exports = {
    "buyer_concentration_2025.csv": concentration_df,
    "buyer_supplier_cpc5_recurrence_2025.csv": recurrence_df,
    "integrated_relationship_signals_2025.csv": ri,
    "signal_intersection_summary_2025.csv": cross_summary_df,
    "pattern_summary_2025.csv": pattern_summary_df,
    "concentration_sensitivity_2025.csv": concentration_sensitivity_df,
    "recurrence_sensitivity_2025.csv": recurrence_sensitivity_df,
    "prioritized_exploratory_cases_2025.csv": case_candidates_df,
    "prioritized_case_procedures_2025.csv": case_procedures_df,
    "exploratory_thresholds_2025.csv": screening_thresholds_df,
    "thresholds_used_2025.csv": thresholds_used_df,
    "signal_coverage_2025.csv": coverage_summary_df,
    "single_bidder_sensitivity_2025.csv": single_bidder_sensitivity_df,
    "single_bidder_by_frequency_2025.csv": single_bidder_frequency_df,
    "prioritization_funnel_2025.csv": prioritization_funnel_df,
}

for filename, df in exports.items():
    path = ANALYTICS_DIR / filename

    df.to_csv(
        path,
        index=False,
        encoding="utf-8-sig",
    )

    print(
        f"{filename}: {len(df):,} filas"
    )
